# 05b · Model training: remaining folds 1-4 (Kaggle, GPU)

Same as 05, but runs folds 1-4. Together with 05 (fold 0) it gives a 5-fold ensemble and out-of-fold predictions for all 58 gold studies.

**Model (`src/model.py`):**
```
2.5D windows from every slot [N,3,224,224]
   → ConvNeXt-Tiny (ImageNet-22k pretrained)          → features [N,768]
   + slot embedding (plane × sequence) + position embedding (where in the stack)
   → 1-layer Transformer (context across windows)
   → 12-head per-finding attention pooling + mean-pooling residual
   → 12 logits
```

**Training (`src/engine.py`):** BCE on soft labels, 2× weight on gold studies, AdamW, cosine learning rate
with warm-up, fp16 AMP, gradient clipping. Augmentation: small per-slot rotation / scale / shift + brightness /
contrast. **No horizontal flip** (it would swap medial and lateral).

**Validation:** every epoch reports `pseudo_auc` (against the report labels of the validation fold) and
`gold_auc` (the ~12 gold studies in the fold). The best epoch is picked by `pseudo_auc`; `gold_auc` is very
noisy on one fold and is read on the 58-study OOF once all five folds are done.

**Kaggle settings:** Accelerator = GPU (T4 / P100), Internet = **on** (to download pretrained weights),
Input = `rsna-knee-src` + the output of 04 (cache).

In [ ]:
# Add the project root (the folder that contains src/) to the Python path; works locally and on Kaggle.
import os, sys
from pathlib import Path

def add_project_root():
    cands = [Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        # os.walk with pruning: never enters the competition DICOM folders (rglob took minutes)
        for d, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
            if Path(d).name == "src" and "__init__.py" in files:
                cands.append(Path(d).parent)
    for c in cands:
        if (c / "src" / "__init__.py").exists():
            sys.path.insert(0, str(c))
            return c
    raise FileNotFoundError("src/ not found")

PROJECT_ROOT = add_project_root()
print("Project root:", PROJECT_ROOT)

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from src.config import EXPERIMENTS_DIR, IS_KAGGLE, TARGET_COLS
from src.dataset import KneeCacheDataset, collate_studies
from src.engine import cosine_with_warmup, evaluate, make_folds, predict, train_one_epoch
from src.model import build_model

CFG = {
    "backbone": "convnext_tiny.fb_in22k_ft_in1k",
    "context_layers": 1,
    "drop_path": 0.1,
    "epochs": 8,
    "batch_size": 2,          # studies per step (~58 windows; fits a 16 GB T4)
    "grad_accum": 4,          # effective batch = 8 studies
    "lr_backbone": 1e-4,
    "lr_head": 5e-4,
    "weight_decay": 0.05,
    "warmup_frac": 0.1,
    "windows_per_slot": 6,    # random windows per slot during training
    "eval_windows_per_slot": 14,
    "gold_weight": 2.0,
    "n_folds": 5,
    "folds_to_run": [1, 2, 3, 4],  # 05b: kalan foldlar (fold 0 = 05)
    "seed": 42,
    "num_workers": 4,
}
DRY_RUN = not IS_KAGGLE
if DRY_RUN:  # local dry run: small backbone, 1 epoch, 2 folds
    CFG.update(backbone="resnet18", n_folds=2, epochs=1, batch_size=2, grad_accum=1, num_workers=0,
               windows_per_slot=2, eval_windows_per_slot=4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"])
print("Cihaz:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "")

## 1. Find the inputs: cache folder and label table

In [ ]:
from src.config import find_input

def find_one(name: str, roots):
    for r in roots:
        if Path(r).exists():
            hit = find_input(name, str(r))   # search that never enters the DICOM folders
            if hit is not None:
                return hit
    raise FileNotFoundError(name)

roots = ["/kaggle/input"] if IS_KAGGLE else [EXPERIMENTS_DIR / "cache", PROJECT_ROOT / "data"]
CACHE_DIR = find_one("cache_config.json", roots).parent
LABELS_PATH = find_one("train_labels_v2.csv", roots)  # labels v2 (gold AUC 0.9014)
cache_cfg = json.loads((CACHE_DIR / "cache_config.json").read_text())
print("Cache   :", CACHE_DIR, cache_cfg)
print("Etiket  :", LABELS_PATH)

labels = pd.read_csv(LABELS_PATH)
if DRY_RUN:
    # Locally the cache holds a single test study. To check that the code runs end to end
    # we copy it under the names of 12 training studies (6 of them gold). The metrics are meaningless.
    import shutil
    src_npz = sorted(CACHE_DIR.glob("*.npz"))[0]
    fake_dir = EXPERIMENTS_DIR / "dryrun_cache"
    fake_dir.mkdir(exist_ok=True)
    labels = pd.concat([labels[labels.is_gold].head(6), labels[~labels.is_gold].head(6)])
    for s in labels.StudyInstanceUID:
        shutil.copy(src_npz, fake_dir / f"{s}.npz")
    CACHE_DIR = fake_dir
cached = {p.stem for p in CACHE_DIR.glob("*.npz")}
missing = (~labels.StudyInstanceUID.isin(cached)).sum()
print(f"Etiketli vaka: {len(labels)} | cache'te olmayan: {missing}")
labels = labels[labels.StudyInstanceUID.isin(cached)].reset_index(drop=True)
folds = make_folds(labels, CFG["n_folds"], CFG["seed"])
folds.groupby("fold").agg(vaka=("StudyInstanceUID", "size"), altin=("is_gold", "sum"))

## 2. Training loop

Separate learning rates for backbone and head: the pretrained backbone moves in small steps (1e-4), the attention
and classifier layers that start from scratch in larger steps (5e-4).

In [ ]:
OUT = EXPERIMENTS_DIR / "model_out"
OUT.mkdir(parents=True, exist_ok=True)

def run_fold(fold: int) -> dict:
    tr = folds[folds.fold != fold].reset_index(drop=True)
    va = folds[folds.fold == fold].reset_index(drop=True)
    dl_tr = DataLoader(KneeCacheDataset(tr, CACHE_DIR, train=True, windows_per_slot=CFG["windows_per_slot"]),
                       batch_size=CFG["batch_size"], shuffle=True, num_workers=CFG["num_workers"],
                       collate_fn=collate_studies, pin_memory=True, drop_last=True,
                       persistent_workers=CFG["num_workers"] > 0)
    dl_va = DataLoader(KneeCacheDataset(va, CACHE_DIR, train=False, eval_windows_per_slot=CFG["eval_windows_per_slot"]),
                       batch_size=CFG["batch_size"], shuffle=False, num_workers=CFG["num_workers"],
                       collate_fn=collate_studies, pin_memory=True)

    model = build_model(CFG, pretrained=True).to(device)
    bb = [p for n, p in model.named_parameters() if n.startswith("backbone.")]
    hd = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    opt = torch.optim.AdamW([{"params": bb, "lr": CFG["lr_backbone"]},
                             {"params": hd, "lr": CFG["lr_head"]}], weight_decay=CFG["weight_decay"])
    steps = CFG["epochs"] * max(1, len(dl_tr) // CFG["grad_accum"])
    sch = cosine_with_warmup(opt, max(1, int(CFG["warmup_frac"] * steps)), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    best, history = -1.0, []
    for ep in range(1, CFG["epochs"] + 1):
        t = time.time()
        loss = train_one_epoch(model, dl_tr, opt, sch, scaler, device, CFG["gold_weight"],
                               CFG["grad_accum"], log_every=100)
        pred = predict(model, dl_va, device)
        m = evaluate(pred, folds)
        history.append({"epoch": ep, "loss": loss, **m, "sec": time.time() - t})
        flag = ""
        score = m["pseudo_auc"] if not np.isnan(m["pseudo_auc"]) else 0.0
        if score > best:
            best = score
            flag = "  <- en iyi, kaydedildi"
            torch.save({"model": model.state_dict(), "config": CFG, "cache_config": cache_cfg,
                        "fold": fold, "epoch": ep, "metrics": m}, OUT / f"model_fold{fold}.pt")
            pred.to_csv(OUT / f"oof_fold{fold}.csv", index=False)
        print(f"fold {fold} ep {ep}: loss {loss:.4f} | pseudo_auc {m['pseudo_auc']:.4f} | "
              f"gold_auc {m['gold_auc']:.4f} (n={m['n_gold']}) | {time.time() - t:.0f}s{flag}", flush=True)
    json.dump(history, open(OUT / f"history_fold{fold}.json", "w"), indent=2)
    del model, opt
    torch.cuda.empty_cache()
    return {"fold": fold, "best_pseudo_auc": best}

results = [run_fold(f) for f in CFG["folds_to_run"]]
pd.DataFrame(results)

## 3. Out-of-fold evaluation

The OOF predictions of all trained folds are merged and the AUC is computed per finding.
The gold AUC is only meaningful once all five folds are done (all 58 gold studies are out-of-fold).

In [ ]:
from src.engine import macro_auc

oof = pd.concat([pd.read_csv(p) for p in sorted(OUT.glob("oof_fold*.csv"))])
m = folds.set_index("StudyInstanceUID").loc[oof.StudyInstanceUID]
per_class = pd.DataFrame({
    "pseudo": macro_auc(m[TARGET_COLS].values, oof[TARGET_COLS].values),
})
g = m.is_gold.values.astype(bool)
if g.sum() >= 5:
    per_class["gold"] = pd.Series(macro_auc(m[TARGET_COLS].values[g], oof[TARGET_COLS].values[g]))
print(f"OOF studies: {len(oof)} | gold: {g.sum()}")
per_class.T.round(3)

**Next:** attach this notebook's output (`model_out/*.pt`) as an input of `06_submission_kaggle.ipynb`.